In [ ]:
#compute G-RIR
import json
import pandas as pd
import numpy as np
import os
import sys
outer_folder = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, outer_folder)
from creme.task_list import TaskList
task_list_instance = TaskList()

#compute G-RIR
def compute_l_rir(orig: float, before: float, after: float, eps: float = 1e-5) -> float:
    gap = orig - before
    if gap < eps: 
        return float('nan')
    return (after - before) / (gap + eps)

def compute_g_rir(l_rir_list: list) -> float:
    rir_array = np.array(l_rir_list)
    valid = rir_array[~np.isnan(rir_array)]
    return float(np.mean(valid)) if len(valid) > 0 else float('nan')

def compute_RIR(task_name,type_case,rtype,test_file="edit_result"):  
    task_list=type_case[rtype]
    df=pd.read_csv(f"{task_name}/{rtype}/{test_file}.csv")
    pass_rates=pd.read_csv(f"{task_name}/diff/{rtype}_pass_rate_differences.csv")
    grir_result=[]
    for task in task_list:
        print(f"\n")
        if "humaneval" in task_name:
            edit_task_id = f"HumanEval/{task}"
        elif "mbpp" in task_name:
            edit_task_id = task
        df_task = df[df["edit_task"] == edit_task_id]
        all_l_rir = []
        for _, row in df_task.iterrows():
            test_task_id = row["task_id"]
            if test_task_id==edit_task_id:
                continue
            after = row["pass_ratio"] 
            raw_row = pass_rates[pass_rates["task_id"] == test_task_id]
            if raw_row.empty:
                continue  
            orig = raw_row["file1_pass_rate"].values[0]/100
            before = raw_row["file2_pass_rate"].values[0]/100
            l_rir = compute_l_rir(orig, before, after)
            all_l_rir.append(l_rir)
            print(f"{test_task_id}: L-RIR = {l_rir:.4f} (orig={orig}, before={before}, after={after})")
        g_rir = compute_g_rir(all_l_rir)
        grir_result.append(g_rir)
        print(f"{edit_task_id} Global Robustness Improvement Ratio (G-RIR): {g_rir:.4f}")
    grir=sum(grir_result)/len(grir_result)
    print(f"total average grir:{grir}")
    return grir

def compute_task_grir(task_name):
    type_case = task_list_instance.get_task_list(task_name)
    print(type_case)
    type_list=["A1","A2","A3","D1","D2","D3","D4","E1","E2","E3","E4","E5","E6","S1","S2",'P1',"P2","C1","C2","C3"]
    test_list=["edit_result"]
    total_result={}
    for test_file in test_list:
        print(f"Computing RIR for {test_file}")
        result={}
        for type_id in type_list:
            result[type_id]=compute_RIR(task_name,type_case,type_id,test_file)
        print(result)
        total_result[test_file]=result
    print(json.dumps(total_result, indent=4, ensure_ascii=False))

# compute_task_grir("humaneval_codellama")
# compute_task_grir("mbpp_codellama")
# compute_task_grir("humaneval_qwen")
compute_task_grir("mbpp_qwen")



In [ ]:
#Key Layer Analysis
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PERT_TYPE_GROUPS = {
    "Additions": ["A2", "A3", "A4"],
    "Deletions": ["D1", "D2", "D3", "D4"],
    "Editing": ["E1", "E2", "E3", "E4", "E5", "E6"],
    "Swap": ["S1", "S2"],
    "Paraphrasing": ["P1", "P2"],
    "Co-occurring": ["C1", "C2", "C3"],
}
def map_pert_type_to_group(pert_type):
    for group, types in PERT_TYPE_GROUPS.items():
        if pert_type in types:
            return group
    return "unknown"
def visualize_hot(csv_path):
    df = pd.read_csv(csv_path)
    df["pert_type_group"] = df["pert_type"].apply(map_pert_type_to_group)
    df["pert_type_group"] = pd.Categorical(df["pert_type_group"], categories=[
                                           "Additions", "Deletions", "Editing", "Swap", "Paraphrasing", "Co-occurring"], ordered=True)
    codellama_df = df[df["model"] == "CodeLLaMA"]
    qwen_df = df[df["model"] == "Qwen"]
    codellama_group_layer_counts = codellama_df.groupby(
        ["pert_type_group", "key_layer"]).size().unstack(fill_value=0)
    qwen_group_layer_counts = qwen_df.groupby(
        ["pert_type_group", "key_layer"]).size().unstack(fill_value=0)
    fig, axes = plt.subplots(2, 1, figsize=(15, 6)) 
    sns.heatmap(codellama_group_layer_counts, cmap="Blues", annot=True,
                fmt="d", linewidths=.5, ax=axes[0], annot_kws={"fontsize": 14})
    axes[0].set_xlabel("Layer (CodeLLama)", fontsize=16)
    axes[0].set_ylabel("Perturbation Type Group", fontsize=16)
    sns.heatmap(qwen_group_layer_counts, cmap="Oranges", annot=True,
                fmt="d", linewidths=.5, ax=axes[1], annot_kws={"fontsize": 14})
    axes[1].set_xlabel("Layer (QWenCoder)", fontsize=16)
    axes[1].set_ylabel("Perturbation Type Group", fontsize=16)
    for ax in axes:
        ax.tick_params(axis='both', labelsize=14)
        ax.set_xticklabels(ax.get_xticklabels(), fontsize=14)
        ax.set_yticklabels(ax.get_yticklabels(), fontsize=14)
    cbar = axes[0].collections[0].colorbar
    cbar.ax.tick_params(labelsize=14)
    cbar = axes[1].collections[0].colorbar
    cbar.ax.tick_params(labelsize=14)
    plt.tight_layout(rect=[0, 0, 1.12, 1])
    plt.show()


visualize_hot("key_layer/key_layer_results.csv")